# Demand Forecasting & Sales Prediction

This notebook develops a machine learning pipeline for predicting retail demand.

## Objectives

- Prepare time-series sales data
- Perform exploratory data analysis
- Engineer temporal and lag-based features
- Select important features
- Train multiple regression models
- Compare model performance
- Select the best-performing model
- Analyze residuals and learning curves
- Save the final model
- Generate predictions for Power BI

In [ ]:
# ============================================
# 1. IMPORT LIBRARIES
# ============================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

import joblib
import os

print("Libraries imported successfully.")

## 2. Load Feature-Engineered Data

In [ ]:
# ============================================
# 2. LOAD FEATURE-ENGINEERED DATA
# ============================================

from pathlib import Path

BASE_DIR = Path.cwd().parent

print("Project directory:")
print(BASE_DIR)

print("\nData directories:")
print(list((BASE_DIR / "data").glob("*")))

In [ ]:
# Find all CSV files in the project

csv_files = list(BASE_DIR.rglob("*.csv"))

for file in csv_files:
    print(file)

In [ ]:
orders_features = pd.read_csv(
    BASE_DIR / "data" / "processed" / "orders_features.csv"
)

print("Shape:", orders_features.shape)
orders_features.head()

In [ ]:
print("Columns in orders_features:")
print(orders_features.columns.tolist())

In [ ]:
orders_features.info()

In [ ]:
orders_features.describe(include="all").T

In [ ]:
orders_features.columns.tolist()

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from pathlib import Path
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

orders_features = pd.read_csv(PROCESSED_DIR / "orders_features.csv")


In [ ]:
df.head()


In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

In [ ]:
df["order_purchase_timestamp"].dtype

In [ ]:
daily_demand = (
    df.groupby(
        df["order_purchase_timestamp"].dt.date
    )
    .size()
    .reset_index(name="order_count")
)

In [ ]:
daily_demand.head(10)

In [ ]:
daily_demand["date"] = pd.to_datetime(
    daily_demand["order_purchase_timestamp"]
)

daily_demand.drop(
    columns=["order_purchase_timestamp"],
    inplace=True
)

In [ ]:
daily_demand.head()

In [ ]:
daily_demand["year"] = daily_demand["date"].dt.year

daily_demand["month"] = daily_demand["date"].dt.month

daily_demand["quarter"] = daily_demand["date"].dt.quarter

daily_demand["day"] = daily_demand["date"].dt.day

daily_demand["weekday"] = daily_demand["date"].dt.weekday

daily_demand["weekend"] = (
    daily_demand["weekday"] >= 5
).astype(int)

In [ ]:
daily_demand.head()

This is feature engineering specifically for forecasting. It doesn't duplicate the features you've already created.

In [ ]:
daily_demand["lag_1"] = daily_demand["order_count"].shift(1)

daily_demand["lag_7"] = daily_demand["order_count"].shift(7)

daily_demand["lag_14"] = daily_demand["order_count"].shift(14)

daily_demand["lag_30"] = daily_demand["order_count"].shift(30)

In [ ]:
daily_demand["rolling_7"] = (
    daily_demand["order_count"]
    .shift(1)
    .rolling(7)
    .mean()
)

daily_demand["rolling_30"] = (
    daily_demand["order_count"]
    .shift(1)
    .rolling(30)
    .mean()
)

In [ ]:
daily_demand = daily_demand.dropna().reset_index(drop=True)

In [ ]:
daily_demand.shape

In [ ]:
daily_demand.head()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    daily_demand["date"],
    daily_demand["order_count"]
)

plt.title("Daily Order Demand")
plt.xlabel("Date")
plt.ylabel("Number of Orders")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
y = daily_demand["order_count"]

In [ ]:
y = daily_demand["order_count"]

In [ ]:
features = [
    "year",
    "month",
    "quarter",
    "day",
    "weekday",
    "weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_7",
    "rolling_30"
]

X = daily_demand[features]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
split_index = int(len(daily_demand) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [ ]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42
    )
}

In [ ]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        predictions
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

In [ ]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    by="RMSE"
)

In [ ]:
best_model_name = (
    results_df
    .sort_values("RMSE")
    .iloc[0]["Model"]
)

print("Best Model:", best_model_name)

In [ ]:
best_model = models[best_model_name]

In [ ]:
best_predictions = best_model.predict(X_test)

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": best_predictions
})

comparison.head(10)

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    y_test.values,
    label="Actual"
)

plt.plot(
    best_predictions,
    label="Predicted"
)

plt.title(
    f"Actual vs Predicted Demand - {best_model_name}"
)

plt.xlabel("Test Time Period")
plt.ylabel("Order Count")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": best_model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    importance["Feature"],
    importance["Importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance - Demand Forecasting")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
import joblib
from pathlib import Path

In [ ]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
best_model = models["Random Forest"]

joblib.dump(
    best_model,
    models_dir / "demand_forecasting_random_forest.pkl"
)

print("Model saved successfully!")

In [ ]:
best_predictions = best_model.predict(X_test)

In [ ]:
forecast_comparison = pd.DataFrame({
    "Date": daily_demand.loc[X_test.index, "date"].values,
    "Actual": y_test.values,
    "Predicted": best_predictions
})

forecast_comparison.head(10)

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    forecast_comparison["Date"],
    forecast_comparison["Actual"],
    label="Actual Demand"
)

plt.plot(
    forecast_comparison["Date"],
    forecast_comparison["Predicted"],
    label="Predicted Demand"
)

plt.title("Random Forest — Actual vs Predicted Demand")
plt.xlabel("Date")
plt.ylabel("Number of Orders")

plt.xticks(rotation=45)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": best_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    importance["Feature"],
    importance["Importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
forecast_comparison["Residual"] = (
    forecast_comparison["Actual"]
    - forecast_comparison["Predicted"]
)

In [ ]:
forecast_comparison["Residual"].describe()

In [ ]:
plt.figure(figsize=(12, 5))

plt.scatter(
    forecast_comparison["Predicted"],
    forecast_comparison["Residual"]
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Predicted Demand")
plt.ylabel("Residual")
plt.title("Residual Analysis — Random Forest")

plt.tight_layout()
plt.show()

In [ ]:
final_mae = mean_absolute_error(
    y_test,
    best_predictions
)

final_mse = mean_squared_error(
    y_test,
    best_predictions
)

final_rmse = np.sqrt(final_mse)

final_r2 = r2_score(
    y_test,
    best_predictions
)

print("Final Model: Random Forest")
print(f"MAE  : {final_mae:.2f}")
print(f"MSE  : {final_mse:.2f}")
print(f"RMSE : {final_rmse:.2f}")
print(f"R²   : {final_r2:.4f}")

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
cv_scores = cross_val_score(
    RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    X,
    y,
    cv=tscv,
    scoring="neg_root_mean_squared_error"
)

In [ ]:
cv_rmse = -cv_scores

print("Cross-validation RMSE:")
print(cv_rmse)

print("\nMean CV RMSE:", cv_rmse.mean())

In [ ]:
from sklearn.model_selection import learning_curve

In [ ]:
train_sizes, train_scores, validation_scores = learning_curve(
    RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    X,
    y,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    train_sizes=np.linspace(0.2, 1.0, 5)
)

In [ ]:
train_rmse = -train_scores.mean(axis=1)
validation_rmse = -validation_scores.mean(axis=1)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    train_sizes,
    train_rmse,
    marker="o",
    label="Training RMSE"
)

plt.plot(
    train_sizes,
    validation_rmse,
    marker="o",
    label="Validation RMSE"
)

plt.xlabel("Training Set Size")
plt.ylabel("RMSE")

plt.title("Learning Curve — Random Forest")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
forecast_comparison.to_csv(
    output_dir / "demand_forecast_results.csv",
    index=False
)

In [ ]:
importance.to_csv(
    output_dir / "demand_feature_importance.csv",
    index=False
)

In [ ]:
results_df.to_csv(
    output_dir / "demand_model_comparison.csv",
    index=False
)

In [ ]:
import joblib
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

model_path = models_dir / "random_forest_demand_model.pkl"

joblib.dump(best_model, model_path)

print(f"Model saved successfully at: {model_path}")

In [ ]:
loaded_model = joblib.load(model_path)

print("Random Forest model loaded successfully!")

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

# Select top 8 features
k = min(8, X_train.shape[1])

selector = SelectKBest(
    score_func=f_regression,
    k=k
)

X_train_kbest = selector.fit_transform(X_train, y_train)
X_test_kbest = selector.transform(X_test)

selected_features_kbest = X_train.columns[
    selector.get_support()
]

print("SelectKBest selected features:")
print(selected_features_kbest.tolist())

In [ ]:
kbest_scores = pd.DataFrame({
    "Feature": X_train.columns,
    "Score": selector.scores_,
    "P_Value": selector.pvalues_
})

kbest_scores = kbest_scores.sort_values(
    "Score",
    ascending=False
)

kbest_scores

In [ ]:
from sklearn.feature_selection import mutual_info_regression

mi_scores = mutual_info_regression(
    X_train,
    y_train,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": X_train.columns,
    "MI_Score": mi_scores
})

mi_df = mi_df.sort_values(
    "MI_Score",
    ascending=False
)

mi_df

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_selector = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_selector.fit(X_train, y_train)

In [ ]:
rf_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_selector.feature_importances_
})

rf_importance = rf_importance.sort_values(
    "Importance",
    ascending=False
)

rf_importance

In [ ]:
import matplotlib.pyplot as plt

top_features = rf_importance.head(12).sort_values(
    "Importance"
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")

plt.tight_layout()
plt.show()

In [ ]:
feature_comparison = pd.DataFrame({
    "Feature": X_train.columns,
    "SelectKBest_Score": selector.scores_,
    "Mutual_Information": mi_scores,
    "Random_Forest_Importance": rf_selector.feature_importances_
})

feature_comparison["KBest_Rank"] = (
    feature_comparison["SelectKBest_Score"]
    .rank(ascending=False)
)

feature_comparison["MI_Rank"] = (
    feature_comparison["Mutual_Information"]
    .rank(ascending=False)
)

feature_comparison["RF_Rank"] = (
    feature_comparison["Random_Forest_Importance"]
    .rank(ascending=False)
)

feature_comparison["Average_Rank"] = (
    feature_comparison[
        ["KBest_Rank", "MI_Rank", "RF_Rank"]
    ].mean(axis=1)
)

feature_comparison = feature_comparison.sort_values(
    "Average_Rank"
)

feature_comparison

In [ ]:
final_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_7",
    "rolling_30"
]

print("Final candidate features:")
print(final_features)

In [ ]:
X_train_selected = X_train[final_features]
X_test_selected = X_test[final_features]

print("Original X_train shape:", X_train.shape)
print("Selected X_train shape:", X_train_selected.shape)

print("\nOriginal feature count:", X_train.shape[1])
print("Selected feature count:", X_train_selected.shape[1])

In [ ]:
rf_selected = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_selected.fit(
    X_train_selected,
    y_train
)

In [ ]:
y_pred_selected = rf_selected.predict(
    X_test_selected
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae_selected = mean_absolute_error(
    y_test,
    y_pred_selected
)

mse_selected = mean_squared_error(
    y_test,
    y_pred_selected
)

rmse_selected = np.sqrt(mse_selected)

r2_selected = r2_score(
    y_test,
    y_pred_selected
)

print("Random Forest — Selected Features")
print("----------------------------------")
print("MAE :", mae_selected)
print("MSE :", mse_selected)
print("RMSE:", rmse_selected)
print("R²  :", r2_selected)

In [ ]:
from pathlib import Path
import joblib
from sklearn.ensemble import RandomForestRegressor

# 1. Define and train your model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 2. Define save directory and create path FIRST
save_path = Path("../models/random_forest_demand_forecasting.pkl")
save_path.parent.mkdir(parents=True, exist_ok=True)

# 3. Save the model
joblib.dump(rf_model, save_path, compress=3)
print(f"Model successfully saved to {save_path.resolve()}")

In [ ]:
final_features = [
    'year',
    'month',
    'quarter',
    'day',
    'weekday',
    'weekend',
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_30',
    'rolling_7',
    'rolling_30'
]

joblib.dump(
    final_features,
    "../models/demand_forecasting_features.pkl"
)

print("Feature list saved successfully.")

In [ ]:
final_metrics = {
    "Model": "Random Forest Regressor",
    "MAE": 41.685289,
    "MSE": 2873.064515,
    "RMSE": 53.600975,
    "R2": 0.688523
}

joblib.dump(
    final_metrics,
    "../models/demand_forecasting_metrics.pkl"
)

print("Model metrics saved successfully.")

In [ ]:
feature_importance_df = pd.DataFrame({
    "Feature": final_features,
    "Importance": rf_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

feature_importance_df

In [ ]:
feature_importance_df.to_csv(
    "../outputs/rf_feature_importance.csv",
    index=False
)

print("Feature importance saved.")

In [ ]:
print(forecast_comparison.columns.tolist())
print(forecast_comparison.shape)
forecast_comparison.head()

In [ ]:
import os

os.makedirs("../outputs/ml", exist_ok=True)

forecast_output = forecast_comparison.copy()

forecast_output.to_csv(
    "../outputs/ml/demand_forecast.csv",
    index=False
)

print("Demand forecast saved successfully.")
print("Shape:", forecast_output.shape)

In [ ]:
results_df.to_csv(
    "../outputs/ml/model_comparison.csv",
    index=False
)

print("Model comparison saved successfully.")

In [ ]:
%whos

In [ ]:
print(forecast_comparison.columns.tolist())
print(forecast_comparison.shape)
forecast_comparison.head()